# An embedding receives learning through several paths

Chapter 2 · Day 2

Input IDs are categorical addresses. The trainable row they select receives gradients, and a tied output table may receive additional dense classification gradients. A zero prompt loss mask does not remove the prompt from the answer's causal history.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms').exists())
sys.path.insert(0, str(ROOT / 'src'))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(909)
plt.rcParams.update({'figure.figsize': (8, 3.6), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})

### Prediction

Prediction: if ID2 appears twice in a batch, are there two independent parameter rows or two contributions to the same row? If prompt loss is zero, must the prompt embedding gradient be zero?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Prediction

Intervention: detach the prompt representation before attention in a copy of the implementation.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
from dongxi_llms.embedding_gradient_lab import repeated_lookup_demo, classifier_path_demo, response_only_loss_demo
lookup = repeated_lookup_demo()
paths = classifier_path_demo()
prompt = response_only_loss_demo()
print('Repeated lookup:', json.dumps(lookup, indent=2))
print('Classifier paths:', json.dumps(paths, indent=2))
print('Prompt gradients:', prompt['prompt_row_gradient_norms'])

### Reference explanation

Reference: that cuts the relevant gradient path, which differs from assigning prompt labels zero loss weight. A detached source may still contribute values during the forward pass while receiving no parameter update through that path.

Reference explanation: lookup gradients accumulate into the single shared row indexed by each ID. Tied input/output weights receive the sum of their computational paths. The answer hidden state can read prompt states through causal attention, so answer loss reaches those prompt embedding rows despite zero direct loss at prompt positions.

The reference prints repeated-row accumulation, matched tied/untied classifier paths, and nonzero prompt-row gradients under response-only loss. A prompt-detach intervention is a proposed change to this implementation; the unchanged code establishes the connected-gradient baseline, not execution of the detached variant.

**Exercise modification reference.** Use the printed nonzero prompt-row gradient norms as the connected-path reference. In a copy of `response_only_loss_demo`, replace the prompt rows of `x` by `torch.cat((x[:2].detach(), x[2:]), dim=0)` before constructing scores and hidden states. This preserves the forward tensor values while cutting those rows’ lookup-parameter paths. The function is an identity-projection attention microscope; the unchanged reference does not execute this detached variant.


### Prediction

Deep checkpoint: is a large gradient row necessarily the most semantically important token?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
norms = prompt['embedding_gradient_norms']
fig, ax = plt.subplots()
ax.bar(range(len(norms)), norms, color='#3265ad')
ax.set(xlabel='Embedding vocabulary row', ylabel='Gradient L2 norm',
       title='Measured gradient from response-only loss')
plt.show()
assert prompt['prompt_row_gradient_norms']['row_1'] > 0
assert prompt['prompt_row_gradient_norms']['row_2'] > 0

### Reference explanation

Reference: this norm depends on the example, objective and parameterization. Its meaning is local sensitivity of this loss; semantic importance requires a different claim and intervention.


**Saved reference preview — rerun the preceding plot cell for current inputs.**

![03 embedding gradient paths: reference plot 01](../figures/chapter-02/day-02-03_embedding_gradient_paths-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.